# Intoduction
From the model comparison notebook, the results were:

- Ridge: RMSE = 0.1411 ± 0.0224
- Lasso: RMSE = 0.1371 ± 0.0271
- ElasticNet: RMSE = 0.1501 ± 0.0254
- rfr: RMSE = 0.1429 ± 0.0083


ElasticNet has the worst RMSE, despite having added complexity than Ridge. Lasso has the worst uncertainty, indicating high instability. These models underperform at baseline levels, so they will not be involved in this notebook


In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

In [2]:
import numpy as np

from src.data import load_train_data
from src.features import add_engineered_features

data = load_train_data()
X = add_engineered_features(data)
y = np.log1p(data['SalePrice'])

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

nominal_cols = ['MSZoning','MSSubClass', 'Street',  'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType','Foundation', 'Heating','CentralAir', 'Electrical','Functional',
        'GarageFinish','PavedDrive',
       'SaleType', 'SaleCondition'
]

ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'HeatingQC','KitchenQual'

]
num_structured_missing = ['GarageYrBlt']
num_unstruc_missing = ['LotFrontage']

rest_num_cols = [
  col for col in X.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

# pipelines

# for numerical attributes with structured missingness, value 0 will be imputed 
num_struc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='constant',
   fill_value = 0)),
  ('scaler',StandardScaler())
])

# for numerical attributes with true missingness, and the rest of the num cols, the median value will be imputed 
rest_num = Pipeline([
  ('imputer',
   SimpleImputer(strategy='median')),
   ('scaler',StandardScaler())
])

# Only few columns need encoding now 
Qual_and_Con_scale = ['None','Po','Fa','TA','Gd','Ex']
qnc_cols = ['ExterQual','ExterCond','HeatingQC','KitchenQual']

# Generate mappings
qnc_map = {k: i for i, k in enumerate(Qual_and_Con_scale)}

# apply the mappings
for col in qnc_cols:
  X[col] = X[col].map(qnc_map)

# impute ordinal (dummy)
ordinal_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent'))
])

# preprocessor
preprocessor = ColumnTransformer([
  ('num_struc',num_struc_missing_pipeline,
    num_structured_missing),
  ('rest_num', rest_num, 
   rest_num_cols),
  ('nominal', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
  ('ordinal',ordinal_pipeline,ordinal_cols)
])

# GridSearch on Ridge

In [5]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
  ('preprocessor', preprocessor),
  ('model',Ridge())
])

param_grid = {
  'model__alpha' : [0.01, 0.1, 1.0, 10.0, 100.0] #in magnitudes
}

grid = GridSearchCV(
  estimator=pipeline,
  param_grid=param_grid,

  scoring="neg_root_mean_squared_error",
  cv=5,
  n_jobs=-1
)

grid.fit(X,y)

print("Best alpha:", grid.best_params_)
print("Best RMSE:",-grid.best_score_)

Best alpha: {'model__alpha': 10.0}
Best RMSE: 0.13656776717556593
